<a href="https://colab.research.google.com/github/iav2002/Assignment_Advanced_Topics_In_DeepLearning/blob/main/Part2_Experiments_4_ViT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Experiment 4 — ViT vs CNN with LoRA

For the final part of 2 we will compare a ViT and a CNN backbone under LoRA fine tuning. So far Exp 2 covered LoRA on ResNet50 across six ranks. The natural way to make this comparison fair is to run the same sweep on a ViT and put the two curves side by side. But will depend on the timing how many rankings we'll perform

A small implementation note before starting. Torchvision's `vit_b_16` builds attention with `nn.MultiheadAttention`, which fuses Q, K and V into a single `in_proj_weight`. `peft` cannot wrap a fused weight, so the cleanest target available is each block's `out_proj`. It still counts as adapting an attention weight, which is what the rubric asks for, and it avoids rewriting the encoder blocks just to separate Q/V. Worth flagging as a deviation from the canonical "Q,V only" recipe in the original LoRA paper.

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import json, random, time
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'device: {device}')
if device.type == 'cuda':
    print(f'gpu: {torch.cuda.get_device_name(0)}')

Mounted at /content/drive
device: cuda
gpu: NVIDIA A100-SXM4-40GB


In [ ]:
# install peft - not in colab by default
!pip install -q -U peft torchao
import peft
print(f'peft version: {peft.__version__}')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 104.8 MB/s eta 0:00:00


peft version: 0.19.1


## 2. Paths and shared training config

Same training config as the previous experiments so the ViT vs ResNet comparison stays clean. The ranks list starts with just `r=8` because we want to time one full ViT epoch before deciding how wide to sweep.

In [ ]:
DRIVE_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AdvancedDL')
INDEX_DIR = DRIVE_ROOT / 'sample_index'
RESULTS_DIR = DRIVE_ROOT / 'results' / 'exp4_vit_lora'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# matches exp1-3 for direct comparability
IMG_SIZE = 224
BATCH_SIZE = 128
NUM_WORKERS = 0
NUM_EPOCHS = 20
LR = 1e-3                # LoRA matrices are random init like a head — needs the higher LR
WEIGHT_DECAY = 1e-4
PATIENCE = 5

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

# starts with just r=8 for smoke test, will extend after timing
RANKS = [8]

print(f'results dir: {RESULTS_DIR}')

results dir: /content/drive/MyDrive/Colab Notebooks/AdvancedDL/results/exp4_vit_lora


## 3. Copy dataset to local SSD and load sample index

Same pipeline used in Exp 1 to 3. Images get copied to local SSD once per session, and the bbox info comes from the JSON sample indexes produced during EDA.

Paths are rewritten so the rest of the notebook reads from local disk instead of Drive.

In [ ]:
import shutil
# 8min L4
# 4min A100

LOCAL_ROOT = Path('/content/dataset_local')
DRIVE_RAW = DRIVE_ROOT / 'raw'
EXPECTED = {'train': 4654, 'val': 1125, 'test': 1124}

def copy_split(split):
    src = DRIVE_RAW / split / 'images'
    dst = LOCAL_ROOT / split / 'images'
    dst.mkdir(parents=True, exist_ok=True)
    if len(list(dst.glob('*.png'))) == EXPECTED[split]:
        print(f'  {split}: already complete')
        return
    print(f'  {split}: copying {EXPECTED[split]} images...')
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f'  {split}: copied in {time.time()-t0:.0f}s')

print('copying dataset to local SSD...')
for split in ['train', 'val', 'test']:
    copy_split(split)
print('done.')

copying dataset to local SSD...
  train: copying 4654 images...
  train: copied in 361s
  val: copying 1125 images...
  val: copied in 86s
  test: copying 1124 images...
  test: copied in 77s
done.


In [ ]:
with open(INDEX_DIR / 'samples_train.json') as f: samples_train = json.load(f)
with open(INDEX_DIR / 'samples_val.json') as f:   samples_val = json.load(f)
with open(INDEX_DIR / 'samples_test.json') as f:  samples_test = json.load(f)
with open(INDEX_DIR / 'class_vocab.json') as f:   vocab = json.load(f)

CLASS_TO_IDX = vocab['class_to_idx']
IDX_TO_CLASS = {int(k): v for k, v in vocab['idx_to_class'].items()}
NUM_CLASSES = len(CLASS_TO_IDX)
CLASSES = vocab['keep_classes']

# rewrite drive paths to local ssd
for split_samples in [samples_train, samples_val, samples_test]:
    for s in split_samples:
        s['img_path'] = s['img_path'].replace(str(DRIVE_RAW), str(LOCAL_ROOT))

print(f'classes: {NUM_CLASSES}')
print(f'train: {len(samples_train):,}  val: {len(samples_val):,}  test: {len(samples_test):,}')

classes: 11
train: 11,000  val: 2,750  test: 2,750


## 4. Pre-load to RAM

Same in memory dataset as before. If the cache file from a previous experiment is still on the local SSD, this should take a few seconds. Otherwise it does the full preload, around three minutes.

In [ ]:
def preload_to_ram(samples):
    n = len(samples)
    imgs = torch.empty((n, 3, IMG_SIZE, IMG_SIZE), dtype=torch.uint8)
    labels = torch.empty((n,), dtype=torch.long)
    t0 = time.time()
    for i, s in enumerate(samples):
        img = Image.open(s['img_path']).convert('RGB')
        x1, y1, x2, y2 = s['bbox']
        crop = img.crop((x1, y1, x2, y2)).resize((IMG_SIZE, IMG_SIZE))
        imgs[i] = torch.from_numpy(np.asarray(crop)).permute(2, 0, 1)
        labels[i] = s['label']
        if (i+1) % 2000 == 0:
            print(f'  {i+1:,}/{n:,}')
    print(f'done: {n:,} in {time.time()-t0:.0f}s')
    return imgs, labels

class InMemoryDataset(Dataset):
    def __init__(self, imgs, labels):
        self.imgs = imgs; self.labels = labels
        # imagenet stats scaled to [0,255] so we apply to uint8 directly
        self.mean = torch.tensor(IMAGENET_MEAN).view(3,1,1) * 255
        self.std = torch.tensor(IMAGENET_STD).view(3,1,1) * 255
    def __len__(self): return len(self.imgs)
    def __getitem__(self, i):
        x = self.imgs[i].float()
        return (x - self.mean) / self.std, self.labels[i]

CACHE_DIR = Path('/content/dataset_local')
def preload_or_load(samples, name):
    path = CACHE_DIR / f'preloaded_{name}.pt'
    if path.exists():
        print(f'{name}: loading cache...')
        cache = torch.load(path)
        return cache['imgs'], cache['labels']
    print(f'{name}: preloading...')
    imgs, labels = preload_to_ram(samples)
    torch.save({'imgs': imgs, 'labels': labels}, path)
    return imgs, labels

train_imgs, train_labels = preload_or_load(samples_train, 'train')
val_imgs, val_labels = preload_or_load(samples_val, 'val')
test_imgs, test_labels = preload_or_load(samples_test, 'test')

train_loader = DataLoader(InMemoryDataset(train_imgs, train_labels), batch_size=BATCH_SIZE, shuffle=True, num_workers=0)
val_loader   = DataLoader(InMemoryDataset(val_imgs, val_labels),     batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(InMemoryDataset(test_imgs, test_labels),   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'\nloaders: train {len(train_loader)} | val {len(val_loader)} | test {len(test_loader)} batches')

train: preloading...


/tmp/ipykernel_1792/1181398063.py:10: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  imgs[i] = torch.from_numpy(np.asarray(crop)).permute(2, 0, 1)


  2,000/11,000
  4,000/11,000
  6,000/11,000
  8,000/11,000
  10,000/11,000
done: 11,000 in 143s
val: preloading...
  2,000/2,750
done: 2,750 in 36s
test: preloading...
  2,000/2,750
done: 2,750 in 36s

loaders: train 86 | val 22 | test 22 batches


## 5. Shared helpers and ViT+LoRA builder

Train and eval loops are the same ones used in earlier experiments. The new piece is `build_vit_lora`, which takes a pretrained ViT and asks `peft` to attach LoRA adapters to every attention block's `out_proj`. The classification head is listed under `modules_to_save` so it trains normally and gets saved together with the adapter.

In [ ]:
import torchvision.models as models
from torch.amp import autocast, GradScaler
from peft import LoraConfig, get_peft_model

def build_base_vit(num_classes):
    m = models.vit_b_16(weights=models.ViT_B_16_Weights.IMAGENET1K_V1)
    # replace 1000-class imagenet head with our 11
    m.heads.head = nn.Linear(m.heads.head.in_features, num_classes)
    return m

def build_vit_lora(num_classes, r):
    base = build_base_vit(num_classes)
    config = LoraConfig(
        r=r,
        lora_alpha=r,                  # alpha=r — same convention as Exp 2
        target_modules=['out_proj'],   # suffix match — hits all 12 attention blocks
        modules_to_save=['head'],      # head trains normally, gets saved with adapter
    )
    return get_peft_model(base, config)

def count_trainable(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total

def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, total_correct, total_n = 0.0, 0, 0
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        total_loss += loss.item() * xb.size(0)
        total_correct += (logits.argmax(1) == yb).sum().item()
        total_n += xb.size(0)
    return total_loss / total_n, total_correct / total_n

@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, total_correct, total_n = 0.0, 0, 0
    # per-class running counters for the per-class accuracy breakdown
    cls_correct = np.zeros(NUM_CLASSES); cls_total = np.zeros(NUM_CLASSES)
    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
        with autocast(device_type='cuda', dtype=torch.float16):
            logits = model(xb)
            loss = criterion(logits, yb)
        preds = logits.argmax(1)
        total_loss += loss.item() * xb.size(0)
        total_correct += (preds == yb).sum().item()
        total_n += xb.size(0)
        for c in range(NUM_CLASSES):
            mask = (yb == c)
            cls_total[c] += mask.sum().item()
            cls_correct[c] += ((preds == yb) & mask).sum().item()
    per_class = (cls_correct / np.maximum(cls_total, 1)).tolist()
    return total_loss / total_n, total_correct / total_n, per_class

## 6. Sanity check on what `peft` is wrapping

Before training anything, build a single ViT+LoRA at r=8 and look at what came out. Two things worth checking: that the trainable parameter count is plausibly small relative to the 86M of full ViT-B/16, and that the adapters actually landed on the modules we expect.

In [ ]:
sanity_model = build_vit_lora(NUM_CLASSES, r=8).to(device)
sanity_model.print_trainable_parameters()

trainable, total = count_trainable(sanity_model)
print(f'\nmanual count: {trainable:,} / {total:,} trainable ({100*trainable/total:.3f}%)')

# confirm peft wrapped where we expect — should see 12 lora adapters (one per encoder block)
lora_modules = [n for n, _ in sanity_model.named_modules() if 'lora_A' in n and 'default' in n]
print(f'\nlora adapters found: {len(lora_modules)}')
print('first 3:', lora_modules[:3])

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 246MB/s]


trainable params: 155,915 || all params: 85,963,030 || trainable%: 0.1814

manual count: 155,915 / 85,963,030 trainable (0.181%)

lora adapters found: 12
first 3: ['base_model.model.encoder.layers.encoder_layer_0.self_attention.out_proj.lora_A.default', 'base_model.model.encoder.layers.encoder_layer_1.self_attention.out_proj.lora_A.default', 'base_model.model.encoder.layers.encoder_layer_2.self_attention.out_proj.lora_A.default']


## 7. Smoke test on epoch timing

ViT-B/16 has roughly three times the parameters of ResNet50 and attention scales differently with sequence length, so the per epoch cost is genuinely unknown until we measure it. The plan is to run one full train epoch plus one val pass, look at the time, and use that to decide how many ranks to include in the sweep. Predicting timing without measuring has been wrong before in this project.

In [ ]:
# decision gate: train + val timing of one epoch decides sweep scope
opt = torch.optim.AdamW(
    [p for p in sanity_model.parameters() if p.requires_grad],
    lr=LR, weight_decay=WEIGHT_DECAY,
)
crit = nn.CrossEntropyLoss()
scaler = GradScaler()

t0 = time.time()
tr_loss, tr_acc = train_one_epoch(sanity_model, train_loader, opt, crit, scaler)
train_time = time.time() - t0

t1 = time.time()
val_loss, val_acc, _ = evaluate(sanity_model, val_loader, crit)
val_time = time.time() - t1

print(f'train epoch: {train_time:.1f}s  (loss {tr_loss:.4f}, acc {tr_acc:.4f})')
print(f'val pass:    {val_time:.1f}s  (loss {val_loss:.4f}, acc {val_acc:.4f})')
print(f'full epoch:  ~{train_time + val_time:.1f}s')
print(f'20 epochs:   ~{20*(train_time + val_time)/60:.1f} min')

# free the smoke test model so the real run starts clean
del sanity_model, opt
torch.cuda.empty_cache()

train epoch: 9.3s  (loss 0.8612, acc 0.7610)
val pass:    1.9s  (loss 0.4484, acc 0.8807)
full epoch:  ~11.3s
20 epochs:   ~3.8 min


## 8. Decision on how many ranks

The smoke test came in at around 11 seconds per epoch on A100, which means a full 20 epoch run takes under 4 minutes. Running the same six ranks as Exp 2 it will take not as expected, so the sweep matches Exp 2 exactly: r=1, 2, 4, 8, 16, 32.

This gives us two parallel curves to compare directly and fair

In [11]:
# matches exp2 ranks exactly so curves overlay cleanly
RANKS = [1, 2, 4, 8, 16, 32]

# trainable param estimate per rank, sanity check before training
print('trainable params by rank:')
for r in RANKS:
    m = build_vit_lora(NUM_CLASSES, r=r)
    t, _ = count_trainable(m)
    print(f'  r={r:2d}: {t:,}')
    del m

trainable params by rank:
  r= 1: 26,891
  r= 2: 45,323
  r= 4: 82,187
  r= 8: 155,915
  r=16: 303,371
  r=32: 598,283


## 9. Training loop over all ranks

Each rank trains independently with early stopping on val loss. Best checkpoint, history JSON and per class accuracy all saved per rank under `results/exp4_vit_lora/lora_rN/`. The structure matches Exp 2 so the comparison plots later can read both experiments the same way.

In [12]:
# expected ~4 min per rank on A100 = ~25 min total
def train_one_rank(r):
    print(f'\n{"="*60}\nrank r={r}\n{"="*60}')
    rank_dir = RESULTS_DIR / f'lora_r{r}'
    rank_dir.mkdir(parents=True, exist_ok=True)

    model = build_vit_lora(NUM_CLASSES, r=r).to(device)
    trainable, total = count_trainable(model)
    print(f'trainable: {trainable:,} / {total:,}')

    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR, weight_decay=WEIGHT_DECAY,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=2)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()

    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': [],
               'lr': [], 'per_class_val_acc': [], 'rank': r, 'trainable_params': trainable}
    best_val_loss = float('inf')
    epochs_no_improve = 0
    t0 = time.time()
    ckpt_path = rank_dir / 'best.pt'

    for epoch in range(NUM_EPOCHS):
        tr_loss, tr_acc = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        val_loss, val_acc, per_class_acc = evaluate(model, val_loader, criterion)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']

        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        history['per_class_val_acc'].append(per_class_acc)

        elapsed = (time.time() - t0) / 60
        print(f'  epoch {epoch:2d} | train {tr_loss:.4f}/{tr_acc:.4f} | '
              f'val {val_loss:.4f}/{val_acc:.4f} | lr {current_lr:.2e} | {elapsed:.1f} min')

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            torch.save({'model_state': model.state_dict(),
                        'epoch': epoch, 'val_loss': val_loss, 'val_acc': val_acc,
                        'rank': r},
                       ckpt_path)
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= PATIENCE:
                print(f'  early stopping at epoch {epoch}')
                break

    history['total_time_min'] = (time.time() - t0) / 60
    with open(rank_dir / 'history.json', 'w') as f:
        json.dump(history, f, indent=2)
    print(f'rank {r} done: {history["total_time_min"]:.1f} min')

    # free gpu memory before next rank
    del model, optimizer
    torch.cuda.empty_cache()
    return history

all_histories = {}
for r in RANKS:
    all_histories[r] = train_one_rank(r)


rank r=1
trainable: 26,891 / 85,834,006
  epoch  0 | train 0.8563/0.7622 | val 0.4494/0.8789 | lr 1.00e-03 | 0.2 min
  epoch  1 | train 0.3638/0.9038 | val 0.3201/0.9105 | lr 1.00e-03 | 0.3 min
  epoch  2 | train 0.2702/0.9291 | val 0.2691/0.9185 | lr 1.00e-03 | 0.5 min
  epoch  3 | train 0.2218/0.9423 | val 0.2300/0.9356 | lr 1.00e-03 | 0.7 min
  epoch  4 | train 0.1903/0.9501 | val 0.2068/0.9389 | lr 1.00e-03 | 0.8 min
  epoch  5 | train 0.1673/0.9567 | val 0.1910/0.9433 | lr 1.00e-03 | 1.0 min
  epoch  6 | train 0.1495/0.9621 | val 0.1789/0.9473 | lr 1.00e-03 | 1.2 min
  epoch  7 | train 0.1355/0.9667 | val 0.1684/0.9505 | lr 1.00e-03 | 1.4 min
  epoch  8 | train 0.1241/0.9694 | val 0.1632/0.9505 | lr 1.00e-03 | 1.6 min
  epoch  9 | train 0.1140/0.9726 | val 0.1542/0.9531 | lr 1.00e-03 | 1.7 min
  epoch 10 | train 0.1059/0.9746 | val 0.1480/0.9553 | lr 1.00e-03 | 1.9 min
  epoch 11 | train 0.0985/0.9766 | val 0.1443/0.9585 | lr 1.00e-03 | 2.1 min
  epoch 12 | train 0.0924/0.9804 | 